# Create and run a local RAG pipeline from scratch

## What is RAG?

**RAG** stands for Retrieval-Augmented Generation.

The goal of RAG is to take information and pass it to an LLM so it can generate outputs based on that info.

* **Retrieval** - Find relevant information given a query, e.g. "what are macronutrients?" -> retrieve passages of text related to macronutrients.
* **Augmented** - We want to take the relevant info from retrieval and augment our input (prompt) to an LLM with that information.
* **Generation** -  Take the first two steps and pass them to an LLM for generative outputs.

## Why RAG?

The main goal of RAG is to improve the generation outputs of LLMs.

1. Prevent hallucinations - LLMs are good at generation good looking text, but that does not mean that it is factually correct.
    * RAG can help LLMs generate information based on relevant passages that are factual
2. Work with custom data - Many base LLMs are trained with internet-scale data (means good understanding with language in general), but means the responses can be generic in nature.
    * RAG helps create specific responses based on specific documents

## Overview of the RAG pipeline

1. Open a PDF document.
2. Format the text of the PDF textbook ready for an embedding model.
3. Embed all the chunks of text in the textbook and turn them into numerical representations which can be stored for use later.
4. Build a retrieval that uses vector search to find relevant chunks of text given a query.
5. Create a prompt that incorporates the retrieved pieces of text.
6. Generate an answer to the query based on the passages of the textbook that were retrieved with an LLM.

**Steps 1-3:** Document preprocessing and embedding creation

**Steps 4-6:** Retrieval and generation (search and answer)

## 1. Document/text processing and embedding creation

Items needed:
* PDF document of choice
* Embedding moodel of choice

Steps:
1. Import the PDF document
2. Process text for embedding (e.g. split into chunks of sentences)
3. Embed text chunks with embedding model
4. Save embeddings to a file for later retrieval

### Import PDF document

In [1]:
import os # file path handling
import requests # download things off the internet

# Get PDF document path
pdf_path = "human-nutrition-text.pdf"

# Download pdf
if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading...")

    # URL of pdf
    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    # local filename
    filename = pdf_path

    # Send a GET request to the URL
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Open the file and save it
        with open(filename, "wb") as file:
            file.write(response.content)
        
        print(f"[INFO] The file has been downloaded and saved as {filename}")
    else:
        print(f"[ERR] Failed to download file. Status code: {response.status_code}")
else:
    print(f"[INFO] File {pdf_path} already exists.")

[INFO] File human-nutrition-text.pdf already exists.


### Open PDF

In [2]:
import fitz # PyMuPDF -> used to read pdf files
from tqdm.auto import tqdm # progress bars

def text_formatter(text: str) -> str:
    """Performs minor formatting on text"""
    
    # remove newlines and extra whitespace chars
    cleaned_text = text.replace("\n", " ").strip()

    return cleaned_text

def read_pdf(path: str) -> list[dict]:
    doc = fitz.open(path)

    pages = []

    # Loop through all the pages in the pdf
    for page_num, page in tqdm(enumerate(doc)):
        text = page.get_text() # get text of current page
        text = text_formatter(text) # format text

        # append all important info about the current page
        pages.append({
            "page_number" : page_num - 41,
            "char_count": len(text),
            "word_count": len(text.split(" ")),
            "sentence_count": len(text.split(". ")),
            "token_count": len(text) / 4, # 1 token ~= 4 chars
            "text": text
        })

    return pages

#### Key term

| Term | Description |
| ----- | ----- | 
| **Token** | A sub-word piece of text. For example, "hello, world!" could be split into ["hello", ",", "world", "!"]. A token can be a whole word,<br> part of a word or group of punctuation characters. 1 token ~= 4 characters in English, 100 tokens ~= 75 words.<br> Text gets broken into tokens before being passed to an LLM. |

In [3]:
pages = read_pdf(path=pdf_path)

import random

random.sample(pages, k=3)

0it [00:00, ?it/s]

[{'page_number': 453,
  'char_count': 387,
  'word_count': 82,
  'sentence_count': 2,
  'token_count': 96.75,
  'text': 'Image  by G.E.  Ulrich, USGS  / CC BY-SA  2.0  Introduction  UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM AND HUMAN NUTRITION PROGRAM  Hoʻā ke ahi, kōʻala ke ola  Light the fire for there is life-giving substance    Learning Objectives  By the end of this chapter, you will be able to:  •  Describe the body’s use, storage and balance of  Introduction  |  453'},
 {'page_number': 123,
  'char_count': 660,
  'word_count': 122,
  'sentence_count': 7,
  'token_count': 165.0,
  'text': 'Image  by Gtirouflet  / CC BY-SA  3.0  dense cortical bone is about 10 percent porous and it looks like  many concentric circles, similar to the rings in a tree trunk,  sandwiched together (Figure 2.27 “Cortical (Compact) Bone”).  Cortical bone tissue makes up approximately 80 percent of the adult  skeleton. It surrounds all trabecular tissue and is the only bone 

In [4]:
import pandas as pd

df = pd.DataFrame(pages) # convert the dict into a dataframe
df.head()

,page_number,char_count,word_count,sentence_count,token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,147,3,199.25,Contents Preface University of Hawai‘i at Mā...


In [5]:
df.describe().round(2) # generate a statistical summary of the data

,page_number,char_count,word_count,sentence_count,token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00
std,348.86,560.38,95.83,6.55,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,5.00,190.50
50%,562.50,1231.50,216.00,10.00,307.88
75%,864.25,1603.50,272.00,15.00,400.88
max,1166.00,2308.00,430.00,39.00,577.00


#### Why does the token count matter?

Token count is important to think about because:
1. Embedding models don't deal with infinite tokens.
2. LLMs don't deal with infinite tokens.

The embedding model we will use (`all-mpnet-base-v2`) is trained to embed sequences of 384 tokens.

As for LLMs, they can't accept infinite tokens in their **context window**.

#### Key term

| Term | Description |
| ----- | ----- | 
| **LLM context window** | The number of tokens a LLM can accept as input. For example, as of March 2024, GPT-4 has a default context window of 32k tokens<br> (about 96 pages of text) but can go up to 128k if needed. A recent open-source LLM from Google, Gemma (March 2024) has a context<br> window of 8,192 tokens (about 24 pages of text). A higher context window means an LLM can accept more relevant information<br> to assist with a query. For example, in a RAG pipeline, if a model has a larger context window, it can accept more reference items<br> from the retrieval system to aid with its generation. |

### Preprocess text for chunking

Split pages into sentences by either:
1. Splitting on `". "`
2. Use NLP library (e.g. `spaCy` and `nltk`)

In [6]:
from spacy.lang.en import English

nlp = English() # create a blank English NLP model

# Add a sentencizer pipeline -> turns text into sentences
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is the first sentence. This is the second sentence. I like cats.")
assert len(list(doc.sents)) == 3

# Print out the sentences split
list(doc.sents)

[This is the first sentence., This is the second sentence., I like cats.]

In [7]:
pages[67]

{'page_number': 26,
 'char_count': 1799,
 'word_count': 321,
 'sentence_count': 21,
 'token_count': 449.75,
 'text': 'Factors that Drive Food Choices  Along with these influences, a number of other factors affect the  dietary choices individuals make, including:  • Taste, texture, and appearance. Individuals have a wide range  of tastes which influence their food choices, leading some to  dislike milk and others to hate raw vegetables. Some foods that  are very healthy, such as tofu, may be unappealing at first to  many people. However, creative cooks can adapt healthy foods  to meet most people’s taste.  • Economics. Access to fresh fruits and vegetables may be scant,  particularly for those who live in economically disadvantaged  or remote areas, where cheaper food options are limited to  convenience stores and fast food.  • Early food experiences. People who were not exposed to  different foods as children, or who were forced to swallow  every last bite of overcooked vegetables, may

In [8]:
for item in tqdm(pages):
    # Apply the pipeline to the text and get the sentences
    item["sentences"] = list(nlp(item["text"]).sents)

    # Convert all sentences to strings (default is a spaCy datatype)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [9]:
random.sample(pages, k=1)

[{'page_number': 1003,
  'char_count': 1307,
  'word_count': 221,
  'sentence_count': 14,
  'token_count': 326.75,
  'text': 'Giardia lamblia is another parasite that is found in contaminated  drinking water. In addition, it lives in the intestinal tracts of animals,  and can wash into surface water and reservoirs, similar to  Cryptosporidium. Giardia causes giardiasis, with symptoms that  include abdominal cramping and diarrhea within one to three days.  Although most people recover within one to two weeks, the disease  can lead to a chronic condition, especially in people with  compromised immune systems.  The  parasite  Toxoplasma  gondii  causes  the  infection  toxoplasmosis, which is a leading cause of death attributed to  foodborne illness in the United States. More than sixty million  Americans carry Toxoplasma gondii, but very few have symptoms.  Typically, the body’s immune system keeps the parasite from  causing disease. Sources include raw or undercooked meat and  unwashed 

In [10]:
df = pd.DataFrame(pages)
df.describe().round(2)

,page_number,char_count,word_count,sentence_count,token_count,sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32
std,348.86,560.38,95.83,6.55,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00


### Chunking

The concept of splitting larger pieces of text into smaller ones is often referred to as text splitting or chunking.

We do this because:
1. Texts are easier to filter (smaller groups are easier to inspect)
2. So text chunks can fit into the embedding model context window (e.g. 384 token limit)
3. The contexts passed to the LLM can be more specific and focused

We will split it into groups of 10 sentences (could also try 5, 7, 8, etc.)

Libraries such as `LangChain` could help with this.

In [11]:
# Define split size to turn groups of sentences into chunks
chunk_size = 10

# Create a function that splits lists of texts recursively into chunk size
# e.g. [20] -> [10, 10]
# e.g. [25] -> [10, 10, 5] (basically do chunk_size until we run out)
def split_list(text: list[str], slice_size: int=chunk_size) -> list[list[str]]:
    return [text[i:i+slice_size] for i in range(0, len(text), slice_size)]

# Test
test_list = list(range(25))
split_list(test_list)

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24]]

In [12]:
# Loop through pages and split the sentences into chunks
for item in tqdm(pages):
    # Use the prev. defined function to split the sentences into chunks of 10
    item["sentence_chunks"] = split_list(item["sentences"],
                                         slice_size=chunk_size)
    
    item["num_chunks"] = len(item["sentence_chunks"]) # how many chunks there are

  0%|          | 0/1208 [00:00<?, ?it/s]

In [13]:
random.sample(pages, k=1)

[{'page_number': 787,
  'char_count': 1366,
  'word_count': 260,
  'sentence_count': 11,
  'token_count': 341.5,
  'text': 'Image by Centers for Disease  Control and Prevention (CDC) /  Public Domain  Tools for Change  A pregnancy may  happen unexpectedly.  Therefore, it is important  for all women of  childbearing age to get  400 micrograms of folate  per day prior to  pregnancy and 600  micrograms per day  during pregnancy. Folate,  which is also known as  folic acid, is crucial for the production of DNA and RNA  and the formation of cells. A deficiency can cause  megaloblastic anemia, or the development of abnormal  red blood cells, in pregnant women. It can also have a  profound effect on the unborn baby. Typically, folate  intake has the greatest impact during the first eight  weeks of pregnancy, when the neural tube closes. The  neural tube develops into the fetus’s brain and spinal  cord, and adequate folate reduces the risk of brain  abnormalities and neural tube defects, which

In [14]:
df = pd.DataFrame(pages)
df.describe().round(2)

,page_number,char_count,word_count,sentence_count,token_count,sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,199.50,10.52,287.00,10.32,1.53
std,348.86,560.38,95.83,6.55,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,5.00,190.50,5.00,1.00
50%,562.50,1231.50,216.00,10.00,307.88,10.00,1.00
75%,864.25,1603.50,272.00,15.00,400.88,15.00,2.00
max,1166.00,2308.00,430.00,39.00,577.00,28.00,3.00


### Split each chunk into its own dictionary item

We want to embed each chunk of sentences into its own numerical representation.

Meaning, we can dive specifically into the text sample that is relevant to a query.

In [15]:
import re # regex

# Split each chunk into its own item
chunks = []

# Loop through all the pages
for item in tqdm(pages):
    # Loop through all the chunks for the current page
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}

        # Get current chunk's page number
        chunk_dict["page_number"] = item["page_number"]

        # Join the lists of sentences back into together into a single paragraph
        joined_sentences = "".join(sentence_chunk).replace("  ", " ").strip() # remove extra spaces
        joined_sentences = re.sub(r'\.([A-Z])', r'. \1', joined_sentences) # ".A" -> ". A"
        
        # Add the paragraph into the dict
        chunk_dict["sentence_chunk"] = joined_sentences

        # Add some stats of the chunks
        chunk_dict["chunk_char_count"] = len(joined_sentences)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentences.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentences) / 4 # 1 token = 4 chars

        chunks.append(chunk_dict)

len(chunks) # how many chunks there are in total



  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [16]:
random.sample(chunks, k=1)

[{'page_number': 302,
  'sentence_chunk': 'Indeed, the very things that make fat- rich foods attractive also make them a hindrance to maintaining a healthful diet. 302 | The Role of Lipids in Food',
  'chunk_char_count': 153,
  'chunk_word_count': 28,
  'chunk_token_count': 38.25}]

In [17]:
df = pd.DataFrame(chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,734.10,112.74,183.52
std,347.79,447.51,71.24,111.88
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,45.00,78.75
50%,586.00,745.00,115.00,186.25
75%,890.00,1118.00,173.00,279.50
max,1166.00,1830.00,297.00,457.50


### Filter out short chunks

These chunks may be too short to be useful for embedding and retrieval. We can filter them out by checking the number of tokens in each chunk.

In [ ]:
# Show random chunks with under 30 tokens (mostly contain headers or links)

min_token_len = 30

# Create a filtered df that contains only the short chunks and loop through the rows
for _, r in df[df["chunk_token_count"] <= min_token_len].sample(5).iterrows():
    # Display the current rows token count and its text
    print(f"Chunk token count: {r['chunk_token_count']} | Text: {r['sentence_chunk']}")


Chunk token count: 20.5 | Text: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=283  Alcohol Metabolism | 441
Chunk token count: 21.75 | Text: Advanced nutrition and human metabolism. Boston, MA: Cengage Learning. Molybdenum | 693
Chunk token count: 5.5 | Text: You can Chloride | 193
Chunk token count: 24.5 | Text: view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=354 Phytochemicals | 605
Chunk token count: 3.25 | Text: 828 | Infancy


In [36]:
# Filter the dataframe for rows with under 30 tokens

# convert the custom df with chunks passing the min token limit into a dictionary
chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_len].to_dict(orient="records")

random.sample(chunks_over_min_token_len, k=1)

[{'page_number': 528,
  'sentence_chunk': 'The RDA for vitamin A is considered sufficient to support growth and development, reproduction, vision, and immune system function while maintaining adequate stores (good for four months) in the liver. Table 9.1 Dietary Reference Intakes for Vitamin A Age Group RDA Males and Females mcg RAE/ day UL Infants (0–6 months) 400* 600 Infants (7–12 months) 500* 600 Children (1–3 years) 300 600 Children (4–8 years) 400 900 Children (9–13 years) 600 1,700 Adolescents (14–18 years) Males: 900 2,800 Adolescents (14–18 years) Females: 700 2,800 Adults (> 19 years) Males: 900 3,000 Adults (> 19 years) Females: 700 3,000 *denotes Adequate Intake Source: Source: Dietary Supplement Fact Sheet: Vitamin A. National Institutes of Health, Office of Dietary Supplements. http://ods.od.nih.gov/factsheets/VitaminA-QuickFacts/. Updated September 5, 2012. Accessed October 7, 2017. Dietary Sources of Vitamin A and Beta-Carotene Preformed vitamin A is found only in foods